# Notebook 10 — Memory for the Research Deep Agent

A complete notebook-first walkthrough of memory management for the
`deep-agents-on-foundry` research agent.

## Sections
- **10.1** Memory foundations: checkpointer vs Store vs memory files
- **10.2** Explicit cross-thread memory tools
- **10.3** Memory scoping and policy
- **10.4** Semantic memory retrieval — personalized RAG
- **10.5** Memory files, context engineering, and economics

## Mental model

```text
                         RESEARCH DEEP AGENT
                                |
              +-----------------+-----------------+
              |                 |                 |
         Checkpointer          Store          Memory files
              |                 |                 |
       thread history       cross-thread       prompt-loaded
       + graph state          memories           knowledge
              |                 |                 |
          thread_id          namespace          AGENTS.md
```

- **Checkpointer** — “What happened before in this thread?”
- **Store** — “What useful information should survive across threads?”
- **Memory files** — “What should this agent always know or follow?”

Memory management is not just storage:

```text
memory =
what to save
+ lifetime
+ scope
+ storage
+ retrieval
+ update
+ freshness
+ forgetting
```

## Setup and API inspection

This notebook assumes the project already exposes:
`build_research_agent`, `build_sqlite_checkpointer`, `thread_config`, and `content_text`.

Because Deep Agents and LangGraph APIs evolve, inspect your installed signatures
before using `context_schema`, indexed Stores, or `memory=`.

In [ ]:
import inspect

from deepagents import create_deep_agent
from langgraph.store.memory import InMemoryStore

print("create_deep_agent:")
print(inspect.signature(create_deep_agent))
print()
print("InMemoryStore:")
print(inspect.signature(InMemoryStore))

# 10.1 — Thread memory vs long-term memory

A LangGraph checkpointer already gives the research agent **thread-scoped memory**:
messages, graph state, checkpoints, and interrupts are persisted under a `thread_id`.

Changing `thread_id` should isolate the conversation.

A useful heuristic:

> If the information should disappear when the thread changes, it belongs in
> thread state. If it should remain useful across unrelated threads, consider
> long-term memory.

In [ ]:
from deep_agents_foundry import (
    build_research_agent,
    build_sqlite_checkpointer,
    thread_config,
    content_text,
)

checkpointer = build_sqlite_checkpointer("../data/checkpoints.db")
research_agent = build_research_agent(checkpointer=checkpointer)

In [ ]:
# Thread A: establish a thread-scoped preference.
thread_a = thread_config("memory-demo-a")

result = research_agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "For this conversation, remember that when we discuss Hosted "
                "Agents I care most about identity and RBAC."
            ),
        }]
    },
    config=thread_a,
)

print(content_text(result["messages"][-1]))

In [ ]:
# Same thread: the checkpointer should restore the earlier turn.
result = research_agent.invoke(
    {"messages": [{"role": "user",
                   "content": "What aspect did I say I care most about?"}]},
    config=thread_a,
)

print(content_text(result["messages"][-1]))

In [ ]:
# New thread: it should NOT automatically inherit thread A's transcript.
thread_b = thread_config("memory-demo-b")

result = research_agent.invoke(
    {"messages": [{"role": "user",
                   "content": "What aspect of Hosted Agents do I care most about?"}]},
    config=thread_b,
)

print(content_text(result["messages"][-1]))

### 10.1 takeaway

| Information | Best home |
|---|---|
| Earlier messages in this conversation | Checkpointer |
| Pending interrupt / graph execution state | Checkpointer |
| “Prefer primary sources” | Store |
| Stable analytical framework | Store |
| Current web facts / prices / news | Usually re-research |
| Small always-on operating instructions | Memory file / instructions |

The checkpointer is **not** inferior to long-term memory. It solves a different
lifetime and isolation problem.

# 10.2 — Explicit cross-thread memory tools

A Store gives us infrastructure, but **supplying `store=` does not define a memory
policy**. The agent still needs a way to decide what to save and what to retrieve.

For learning, make memory explicit with two tools:

```text
remember_research_preference
recall_research_preferences
```

We intentionally avoid semantic search and automatic extraction here.

In [ ]:
from uuid import uuid4
from langchain_core.tools import tool
from langgraph.prebuilt import ToolRuntime

store = InMemoryStore()

In [ ]:
@tool
def remember_research_preference(
    preference: str,
    runtime: ToolRuntime,
) -> str:
    """Remember a durable research preference for the current user."""

    if runtime.store is None:
        return "Long-term memory store is unavailable."

    namespace = ("users", "demo-user", "research_preferences")
    memory_id = str(uuid4())

    runtime.store.put(
        namespace,
        memory_id,
        {"text": preference},
    )
    return f"Remembered research preference: {preference}"


@tool
def recall_research_preferences(
    runtime: ToolRuntime,
) -> str:
    """Recall durable research preferences for the current user."""

    if runtime.store is None:
        return "Long-term memory store is unavailable."

    namespace = ("users", "demo-user", "research_preferences")
    memories = runtime.store.search(namespace)

    if not memories:
        return "No research preferences have been saved."

    return "\n".join(f"- {item.value['text']}" for item in memories)

`ToolRuntime` is injected by LangGraph; the model does not choose or manufacture
the runtime object. This is important because infrastructure and identity should
not be model-controlled.

In [ ]:
from deep_agents_foundry.agent import RESEARCH_INSTRUCTIONS
from deep_agents_foundry.model import build_model
from deep_agents_foundry.tools import build_web_search_tool

model = build_model()
web_search = build_web_search_tool()

MEMORY_POLICY = """

Long-term memory policy:
- If the user explicitly asks you to remember a stable research preference,
  call remember_research_preference.
- If the user asks what research preferences have been remembered,
  call recall_research_preferences.
- Do not store changing web facts, news, prices, current product capabilities,
  or other time-sensitive findings as durable preference memory.
"""

memory_agent = create_deep_agent(
    model=model,
    tools=[
        web_search,
        remember_research_preference,
        recall_research_preferences,
    ],
    system_prompt=RESEARCH_INSTRUCTIONS + MEMORY_POLICY,
    checkpointer=checkpointer,
    store=store,
)

In [ ]:
# Thread A writes a durable preference.
result = memory_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": (
            "Remember this for future research: I prefer primary sources, "
            "and I want architecture trade-offs emphasized."
        ),
    }]},
    config=thread_config("memory-cross-thread-a"),
)

print(content_text(result["messages"][-1]))

In [ ]:
# Inspect the memory Store directly.
namespace = ("users", "demo-user", "research_preferences")

for item in store.search(namespace):
    print("key:", item.key)
    print("value:", item.value)
    print()

In [ ]:
# A completely different thread can retrieve the same durable preference.
result = memory_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "What research preferences have you remembered for me?",
    }]},
    config=thread_config("memory-cross-thread-b"),
)

print(content_text(result["messages"][-1]))

The architecture is now:

```text
Thread A
   ↓ Store.put
Long-term Store
   ↓ Store.search
Thread B
```

The two LangGraph threads remain isolated; only selected durable memory crosses
the boundary.

In [ ]:
# Normal web research should not automatically create durable preference memories.
before = len(store.search(namespace))

result = memory_agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Research the latest Microsoft Foundry Hosted Agent capabilities.",
    }]},
    config=thread_config("memory-web-research-test"),
)

after = len(store.search(namespace))

print("memory count before:", before)
print("memory count after :", after)
print("delta              :", after - before)

### 10.2 takeaway

The Store is the easy part. The hard part is **memory policy**.

For a web research agent:

```text
stable user preference
→ strong memory candidate

changing web fact
→ research fresh
```

Avoid turning tool outputs and current facts into permanent memory by default.

# 10.3 — Memory scoping and policy

Three identifiers answer three different questions:

```text
thread_id
→ Which conversation is this?

user_id
→ Whose long-term memory is this?

namespace
→ What kind of memory is this?
```

Never use the model to choose the user identity it should access.

In [ ]:
from dataclasses import dataclass

@dataclass
class ResearchContext:
    user_id: str

A production-quality direction is to pass `user_id` through trusted runtime
context and have tools read it from `runtime.context`.

Inspect your installed Deep Agents signature before wiring `context_schema`.

In [ ]:
print(inspect.signature(create_deep_agent))

In [ ]:
@tool
def remember_research_preference_scoped(
    preference: str,
    runtime: ToolRuntime,
) -> str:
    """Remember a research preference for the runtime user."""

    if runtime.store is None:
        return "Long-term memory store is unavailable."

    user_id = runtime.context.user_id
    namespace = ("users", user_id, "research_preferences")

    runtime.store.put(
        namespace,
        str(uuid4()),
        {"text": preference},
    )
    return f"Remembered research preference: {preference}"


@tool
def recall_research_preferences_scoped(
    runtime: ToolRuntime,
) -> str:
    """Recall preferences for the runtime user."""

    if runtime.store is None:
        return "Long-term memory store is unavailable."

    namespace = (
        "users",
        runtime.context.user_id,
        "research_preferences",
    )

    memories = runtime.store.search(namespace)

    if not memories:
        return "No saved research preferences."

    return "\n".join(f"- {item.value['text']}" for item in memories)

### User isolation experiment

Once the installed API is wired with `ResearchContext`, prove:

```text
User A, Thread 1
   ↓ saves preference

User B, Thread 1
   ↓ cannot see User A's preference

User A, Thread 2
   ↓ can still retrieve User A's preference
```

This proves **thread isolation + user-memory isolation** independently.

## Memory categories and namespaces

A useful shape for this research agent:

```text
("users", user_id, "preferences")
("users", user_id, "research_frameworks")
("projects", project_id, "context")
```

Namespaces help targeted retrieval. A technical architecture task may need
preferences + research frameworks, but not every memory from every project.

## Append vs update vs consolidate

Three strategies:

1. **Append** — preserve each observation as a new memory
2. **Update** — overwrite a known canonical preference
3. **Consolidate** — retrieve related memories and synthesize one canonical memory

A useful hybrid:

```text
structured singleton preference
→ deterministic key

open-ended observation
→ unique ID
```

In [ ]:
structured_namespace = ("users", "demo-user", "preferences")

store.put(
    structured_namespace,
    "source_preference",
    {"text": "Prefer primary sources."},
)

print(store.get(structured_namespace, "source_preference").value)

# Later, update the same logical preference rather than append a duplicate.
store.put(
    structured_namespace,
    "source_preference",
    {"text": "Prefer primary sources, especially official documentation."},
)

print(store.get(structured_namespace, "source_preference").value)

## Conflict and scope precedence

A user may have:

```text
global:
"Generally concise."

project:
"For Deep Agents learning, detailed explanations."

thread:
"For this answer, give me only a summary."
```

A sensible precedence model:

```text
thread instruction
   ↓ overrides
project preference
   ↓ overrides
global preference
```

Memory scoping resembles configuration layering.

## Freshness, confidence, and explicit vs inferred memory

Useful memory metadata can include:

```python
{
    "text": "...",
    "created_at": "...",
    "kind": "preference",
    "source": "explicit_user_instruction",
    "confidence": 1.0,
}
```

Potentially also `expires_at` for temporary memories.

Prefer explicit memory first:

```text
"Remember that I prefer primary sources."
→ high-confidence memory

"Give me a short answer this time."
→ do NOT infer "user always prefers short answers"
```

Inferred memory is powerful but much easier to get wrong.

## Forgetting is part of memory

A real memory system supports:

```text
remember
recall
update
forget
```

not “remember forever.”

In [ ]:
forget_namespace = ("users", "demo-user", "preferences")

store.put(
    forget_namespace,
    "temporary_preference",
    {"text": "This memory will be deleted."},
)

print("before:", store.get(forget_namespace, "temporary_preference"))

store.delete(
    forget_namespace,
    "temporary_preference",
)

print("after :", store.get(forget_namespace, "temporary_preference"))

## Memory decision checklist

For every candidate memory, ask:

1. Is it useful beyond this thread?
2. Is it likely to remain true?
3. Did the user explicitly ask to remember it?
4. Who owns it: user, project, organization, agent?
5. Is there already a related memory that should be updated?
6. When should it be retrieved?
7. When should it expire or be forgotten?

### Recommended policy for this research agent

**Remember**
- explicit stable research preferences
- stable analytical frameworks
- persistent style/format preferences
- stable project objectives when marked durable

**Do not remember by default**
- current web facts
- prices
- news
- current API/model capabilities
- transient research findings
- raw tool outputs
- every conversation detail

# 10.4 — Semantic memory retrieval: personalized RAG

Exact lookup works when the key is known.

Namespace scans work when memory is small.

Semantic retrieval answers:

> Which saved memories are most relevant to this natural-language query?

That is where memory starts to resemble RAG.

Suppose the Store contains:

- Prefer primary sources whenever possible.
- Emphasize architecture trade-offs.
- Keep research concise unless I ask for depth.
- For investing topics, focus on owner economics and FCF/share.

Query:

> “What source types should you prioritize?”

Semantic retrieval can surface the first memory even though the wording differs.

## Prepare an indexed Store

Use the embeddings implementation already available in your Azure/Foundry environment.
Do not introduce a new provider just for this notebook.

Inspect the installed `InMemoryStore` signature first:

In [ ]:
print(inspect.signature(InMemoryStore))

In [ ]:
# Illustrative pattern — adapt to the embeddings client/deployment already
# available in your environment.
#
# embedding_dimension = len(embeddings.embed_query("dimension probe"))
#
# semantic_store = InMemoryStore(
#     index={
#         "embed": embeddings,
#         "dims": embedding_dimension,
#         "fields": ["text"],
#     }
# )

In [ ]:
# Once semantic_store exists:
#
# user_id = "semantic-demo-user"
# preference_namespace = ("users", user_id, "preferences")
#
# semantic_store.put(
#     preference_namespace,
#     "source_preference",
#     {"text": "Prefer primary sources whenever possible."},
# )
# semantic_store.put(
#     preference_namespace,
#     "architecture_focus",
#     {"text": "Emphasize architecture trade-offs in technical research."},
# )
# semantic_store.put(
#     preference_namespace,
#     "style_preference",
#     {"text": "Keep research concise unless I ask for more depth."},
# )
# semantic_store.put(
#     preference_namespace,
#     "investing_framework",
#     {"text": "For investing topics, focus on owner economics, FCF/share, and dilution."},
# )

## The retrieval triad

### Exact key lookup
Best when the desired memory is structured and known.

```python
store.get(namespace, "source_preference")
```

### Namespace scan
Best for inspection or small complete lists.

```python
store.search(namespace)
```

### Semantic retrieval
Best when relevance matters and wording may differ.

```python
store.search(
    namespace,
    query="What sources should you prioritize?",
    limit=3,
)
```

In [ ]:
# Example semantic retrieval:
#
# results = semantic_store.search(
#     preference_namespace,
#     query="What kind of sources should you prioritize?",
#     limit=3,
# )
#
# for item in results:
#     print("key:", item.key)
#     print("value:", item.value)
#     print("score:", getattr(item, "score", None))
#     print()

## Tiny memory-retrieval benchmark

Try queries such as:

```text
"What source types should you use?"
"How should you approach technical system analysis?"
"How detailed should your answers be?"
"What investing lens should you use?"
```

Inspect the top memories and scores.

This is the beginning of a retrieval evaluation dataset for memory.

In [ ]:
# Example:
#
# queries = [
#     "What source types should you use?",
#     "How should you approach technical system analysis?",
#     "How detailed should your answers be?",
#     "What investing lens should you use?",
# ]
#
# for q in queries:
#     print("=" * 80)
#     print("QUERY:", q)
#     results = semantic_store.search(
#         preference_namespace,
#         query=q,
#         limit=2,
#     )
#     for item in results:
#         print(item.key, "->", item.value["text"],
#               "| score:", getattr(item, "score", None))

## Semantic memory ≈ personalized RAG

```text
RAG:
external corpus
→ embed
→ retrieve relevant chunks
→ inject context

Semantic memory:
saved personalized memories
→ embed
→ retrieve relevant memories
→ inject context
```

The distinction:

```text
RAG corpus
= external knowledge

Memory Store
= personalized durable context
```

In [ ]:
@tool
def semantic_recall_research_preferences(
    query: str,
    runtime: ToolRuntime,
) -> str:
    """Recall the most relevant saved research preferences for the current user."""

    if runtime.store is None:
        return "Long-term memory store is unavailable."

    user_id = runtime.context.user_id
    namespace = ("users", user_id, "preferences")

    results = runtime.store.search(
        namespace,
        query=query,
        limit=3,
    )

    if not results:
        return "No relevant research preferences found."

    return "\n".join(
        f"- {item.value['text']}"
        for item in results
    )

## Semantic memory failure modes

These should look familiar from RAG:

- false positives
- false negatives
- over-retrieval
- conflicting memories
- stale memories
- context pollution
- embedding/retrieval cost

So memory retrieval also needs:
**precision, recall, relevance, freshness, context-budget, and cost evaluation**.

Do not retrieve every memory on every turn. Prefer selective retrieval:
- at task start
- when the user asks about preferences
- when personalization clearly affects the task
- when a stable framework is relevant

# 10.5 — Memory files vs Store vs Checkpointer

Deep Agents can also expose a `memory=` mechanism for durable text files such as
`AGENTS.md` (verify the installed API first).

The important distinction:

```text
Store
→ selective / lazy retrieval

Memory file
→ eager / always-loaded context
```

In [ ]:
from pathlib import Path

memory_dir = Path("../memory_files")
memory_dir.mkdir(parents=True, exist_ok=True)

agents_md = memory_dir / "AGENTS.md"

agents_md.write_text(
    """
# Research preferences

- Prefer primary and authoritative sources.
- Explain architecture before implementation details.
- Emphasize trade-offs.
- Keep explanations intuitive unless deeper detail is requested.
""".strip(),
    encoding="utf-8",
)

print(agents_md.read_text())

In [ ]:
# Verify the exact installed Deep Agents contract first.
print(inspect.signature(create_deep_agent))

# Conceptual construction if supported:
#
# memory_file_agent = create_deep_agent(
#     model=model,
#     tools=[web_search],
#     system_prompt=RESEARCH_INSTRUCTIONS,
#     checkpointer=checkpointer,
#     memory=[str(agents_md)],
# )

## Eager vs selective context

### Store

```text
query
  ↓
decide memory is relevant
  ↓
Store.search(...)
  ↓
retrieve only relevant memory
  ↓
use in context
```

### Memory file

```text
agent starts
  ↓
AGENTS.md loaded
  ↓
instructions already in context
  ↓
answer
```

Memory files are useful for:
- small stable operating instructions
- project conventions
- reusable policy
- broadly relevant behavior

They become harmful when they grow into a giant history dump.

## Memory becomes context engineering

Every memory mechanism eventually asks:

> Should this information be in the model context right now?

A practical hierarchy:

```text
LEVEL 1 — System prompt
core identity / immutable research behavior

LEVEL 2 — Memory files
small always-on operating instructions

LEVEL 3 — Store
selective user/project long-term memory

LEVEL 4 — Checkpointer
current conversation and execution state

LEVEL 5 — Workspace/files
temporary working artifacts
```

Context is then assembled from only what is needed.

## Episodic, semantic, and procedural memory

```text
Episodic:
"What happened in this interaction?"
→ checkpointer/thread history

Semantic:
"What stable facts/preferences do I know?"
→ Store

Procedural:
"How should I perform this task?"
→ instructions / memory files / later Skills
```

This creates the bridge to Notebook 11:

```text
Memory:
"What do I know?"

Skill:
"How do I do something?"
```

## Context economics

Illustrative example:

```text
AGENTS.md = 2,000 tokens
research run = 8 model calls
≈ 16,000 repeated input-token contribution
```

Versus selective retrieval:

```text
200 relevant tokens × 3 relevant calls
≈ 600 token contribution
```

These are illustrative numbers, but the architectural trade-off is real:

```text
always-loaded context
→ predictable but potentially expensive

retrieved context
→ efficient but retrieval can fail
```

This is a direct bridge to the later agent-economics capstone.

# Notebook 10 — Final takeaways

1. **Checkpointer** = thread-scoped conversation/execution memory
2. **Store** = cross-thread durable memory
3. **Memory files** = eagerly loaded operating knowledge/instructions
4. `thread_id`, `user_id`, and namespace solve different scoping problems
5. Stable user preferences are good memory candidates
6. Fast-changing web facts should normally be re-researched
7. Memory must support update, conflict handling, freshness, and forgetting
8. Semantic memory is essentially personalized RAG
9. Retrieval quality and context cost matter just as much as storage
10. Memory design is fundamentally **context engineering**

Final architecture:

```text
system prompt
      +
small always-loaded memory
      +
relevant Store memories
      +
current thread
      +
needed workspace artifacts
      ↓
Research Deep Agent
```

## Self-check

You should now be able to explain:
- checkpointer vs Store vs memory file
- `thread_id` vs `user_id`
- namespace design
- append vs update vs consolidate
- explicit vs inferred memory
- freshness and forgetting
- exact vs semantic retrieval
- why semantic memory resembles RAG
- how memory affects context-window economics
- why Skills naturally come next